# Music Annotation Project
## Notebook 75: Frozen V8.8 Collar-Preservation Counterfactual

**Runtime: CPU. Select Runtime > Run all.**

This notebook evaluates a narrow challenger without changing `v8_8_operator.py` or the frozen incumbent outputs.

### Predeclared arms

- **C0_INCUMBENT**: frozen source output unchanged.
- **C1_PRESERVE_RING2**: copy exact observed bytes into dilation Ring 2 only.
- **C2_PRESERVE_RING1_RING2**: copy exact observed bytes into Ring 1 and Ring 2.

The seed, exact dilation-2 prepared mask, V7.8/V8.8 routes, PRINT model outputs, SAFE_REMOVE/M5 decisions and all bytes outside the selected collar remain unchanged.

### Cohorts

- **Paired development**: uses clean truth post hoc to measure genuine printed ink recovered, annotation retained and truth-uncertain pixels retained.
- **HITL annotated-only**: creates production-like C0/C1/C2 pages and change overlays for visual review only. No truth claims are made.

### Truth firewall

All C0/C1/C2 pages and their hashes are written before clean pages are loaded. Clean truth never changes an arm, mask, route or output byte.

### Interpretation

This is a counterfactual development audit, not a production release. A favourable arm remains a challenger until separately frozen and validated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib seaborn scikit-image

## 1. Configuration and run identity

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, zipfile, gc
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.metrics import structural_similarity as ssim

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB74_ROOT=ROOT/'notebook74_combined_mask_annotation_distance_audit'
SOURCE_NB74_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'notebook75_v88_collar_preservation_counterfactual'

RUN_HITL=True
RUN_PAIRED=True
CHANGE_THRESHOLD=8
DARK_THRESHOLD=190
BOOTSTRAP_RESAMPLES=10000
SEED=42
MAX_CONTACT_PAGES=16
MAX_FOCUS_CROPS=48
CROP_MARGIN=90

ARMS=['C0_INCUMBENT','C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2']
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook75_collar_counterfactual'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; PRED=OUT/'prediction_lock'; TABLES=OUT/'tables'; PAGES=OUT/'pages'; VISUAL=OUT/'visuals'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,PRED,TABLES,PAGES,VISUAL,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Helpers and completed Notebook 74 source resolution

In [ ]:
def require(x,msg):
    if not bool(x):raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def object_hash(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp');t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str));t.replace(path)
def read_gray(path):
    a=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE);require(a is not None,f'Unreadable image: {path}');return a
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def geometry(seed):
    k1=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3));k2=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    d1=cv2.dilate(seed.astype(np.uint8),k1)>0;d2=cv2.dilate(seed.astype(np.uint8),k2)>0
    return d1,d2,d1&~seed,d2&~d1

def printed_ink(clean):return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def tolerant_clean_print(clean):return cv2.dilate(printed_ink(clean).astype(np.uint8),np.ones((3,3),np.uint8))>0

def latest_nb74():
    return latest_complete(NB74_ROOT,['inputs/combined_roster.csv','audit/input_resolution_and_geometry.csv','tables/page_summary.csv','pages'])
SOURCE=Path(SOURCE_NB74_RUN_OVERRIDE) if SOURCE_NB74_RUN_OVERRIDE else latest_nb74()
require(SOURCE is not None,'No complete combined Notebook 74 run found.')
roster=pd.read_csv(SOURCE/'inputs/combined_roster.csv',dtype=str).fillna('')
checks=pd.read_csv(SOURCE/'audit/input_resolution_and_geometry.csv')
require((checks.prepared_vs_exact_dilation2_xor==0).all(),'Notebook 74 reported non-exact dilation-2 geometry.')
if not RUN_HITL:roster=roster[roster.population!='HITL_ANNOTATED_ONLY']
if not RUN_PAIRED:roster=roster[roster.population!='PAIRED_DEVELOPMENT']
require(len(roster)>0,'Selected cohort roster is empty.')
shutil.copy2(SOURCE/'inputs/combined_roster.csv',INPUTS/'source_combined_roster.csv')
shutil.copy2(SOURCE/'audit/input_resolution_and_geometry.csv',INPUTS/'source_geometry_audit.csv')
print(roster.groupby('population').size())

## 3. Load frozen incumbent arrays and construct deterministic arms

In [ ]:
def load_page(r,load_clean=False):
    observed=read_gray(r.observed_path);seed=read_gray(r.seed_path)>0
    clean=read_gray(r.clean_path) if load_clean and r.clean_path else None
    if r.population=='HITL_ANNOTATED_ONLY':
        incumbent=read_gray(r.corrected_path);prepared=read_gray(r.prepared_path)>0;reason=read_gray(r.reason_path)
        route=np.select([reason==1,reason==2,reason==3,reason==4],['V78_PRESERVE','STRICT_PRINT','M5_REMOVE','SAFE_REMOVE'],default='OTHER')
    else:
        with np.load(r.cache_path) as z:
            need=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous'];missing=[x for x in need if x not in z.files];require(not missing,f'{r.page_key}: cache missing {missing}')
            prepared=z['mask']>0;incumbent=z['geometry_t80'];pres=z['g80_preserve']>0;rem=z['g80_remove']>0;amb=z['g80_ambiguous']>0
        route=np.select([pres,rem,amb],['V78_PRESERVE','V78_REMOVE','V78_AMBIGUOUS'],default='OTHER')
    require(observed.shape==seed.shape==prepared.shape==incumbent.shape,f'{r.page_key}: shape mismatch')
    if clean is not None:require(clean.shape==observed.shape,f'{r.page_key}: clean shape mismatch')
    d1,d2,ring1,ring2=geometry(seed);require(np.array_equal(prepared,d2),f'{r.page_key}: prepared mask is not exact dilation-2')
    c0=incumbent.copy();c1=c0.copy();c1[ring2]=observed[ring2];c2=c0.copy();c2[ring1|ring2]=observed[ring1|ring2]
    arms={'C0_INCUMBENT':c0,'C1_PRESERVE_RING2':c1,'C2_PRESERVE_RING1_RING2':c2}
    for name,a in arms.items():
        require(np.array_equal(a[~prepared],observed[~prepared]),f'{r.page_key}/{name}: outside-mask drift')
        if name=='C1_PRESERVE_RING2':
            require(np.array_equal(a[ring2],observed[ring2]),f'{r.page_key}: C1 ring2 mismatch');require(np.array_equal(a[~ring2],c0[~ring2]),f'{r.page_key}: C1 non-ring2 regression')
        if name=='C2_PRESERVE_RING1_RING2':
            collar=ring1|ring2;require(np.array_equal(a[collar],observed[collar]),f'{r.page_key}: C2 collar mismatch');require(np.array_equal(a[~collar],c0[~collar]),f'{r.page_key}: C2 non-collar regression')
    return locals()

identity_rows=[]
for _,r in roster.iterrows():
    z=load_page(r,False)
    identity_rows.append({'page_key':r.page_key,'page':r.page,'population':r.population,'source_group':r.source_group,'observed_hash':array_hash(z['observed']),'seed_hash':array_hash(z['seed'].astype(np.uint8)),'prepared_hash':array_hash(z['prepared'].astype(np.uint8)),'ring1_hash':array_hash(z['ring1'].astype(np.uint8)),'ring2_hash':array_hash(z['ring2'].astype(np.uint8)),**{f'{k}_hash':array_hash(v) for k,v in z['arms'].items()}})
    del z;gc.collect()
identity_df=pd.DataFrame(identity_rows);identity_df.to_csv(AUDIT/'arm_semantic_hashes.csv',index=False)
print('Prepared deterministic arms:',len(identity_df))

## 4. Write C0/C1/C2 pages and create the prediction lock

This stage does not load clean pages. It writes every arm and locks semantic hashes before truth scoring.

In [ ]:
page_output_rows=[]
for _,r in roster.iterrows():
    z=load_page(r,False);d=PAGES/r.page_key;d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'observed.png'),z['observed']);cv2.imwrite(str(d/'seed.png'),z['seed'].astype(np.uint8)*255);cv2.imwrite(str(d/'prepared.png'),z['prepared'].astype(np.uint8)*255);cv2.imwrite(str(d/'ring1.png'),z['ring1'].astype(np.uint8)*255);cv2.imwrite(str(d/'ring2.png'),z['ring2'].astype(np.uint8)*255)
    for name,a in z['arms'].items():cv2.imwrite(str(d/f'{name}.png'),a)
    # Exact difference maps against incumbent.
    for name in ['C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2']:
        diff=np.zeros_like(z['observed']);diff[z['arms'][name]!=z['arms']['C0_INCUMBENT']]=255;cv2.imwrite(str(d/f'{name}_vs_C0_difference.png'),diff)
    page_output_rows.append({'page_key':r.page_key,'population':r.population,'source_group':r.source_group,'observed_sha256':sha256_file(d/'observed.png'),'seed_sha256':sha256_file(d/'seed.png'),'prepared_sha256':sha256_file(d/'prepared.png'),**{f'{name}_sha256':sha256_file(d/f'{name}.png') for name in ARMS}})
    del z;gc.collect()
page_outputs=pd.DataFrame(page_output_rows);page_outputs.to_csv(PRED/'page_output_hashes.csv',index=False)
lock_payload={'source_nb74_run':str(SOURCE),'source_roster_sha256':sha256_file(SOURCE/'inputs/combined_roster.csv'),'source_geometry_audit_sha256':sha256_file(SOURCE/'audit/input_resolution_and_geometry.csv'),'arms':ARMS,'arm_rule':{'C0':'incumbent byte-for-byte','C1':'observed bytes in ring2 only','C2':'observed bytes in ring1 and ring2'},'page_output_hashes_sha256':sha256_file(PRED/'page_output_hashes.csv'),'arm_semantic_hashes_sha256':sha256_file(AUDIT/'arm_semantic_hashes.csv'),'pages':len(roster),'truth_loaded':False}
lock={'lock_id':object_hash(lock_payload),'created_utc':datetime.now(timezone.utc).isoformat(),'state':'CLOSED_BEFORE_TRUTH','payload':lock_payload}
atomic_json(PRED/'prediction_lock.json',lock);print('Prediction lock:',lock['lock_id'])

## 5. Post-lock paired truth labels and arm scoring

In [ ]:
# Verify lock before any clean truth is loaded.
lock=json.loads((PRED/'prediction_lock.json').read_text());require(lock['state']=='CLOSED_BEFORE_TRUTH','Prediction lock is not closed.')
require(lock['payload']['page_output_hashes_sha256']==sha256_file(PRED/'page_output_hashes.csv'),'Page outputs changed after lock.')

pixel_rows=[];page_rows=[];component_rows=[]
for _,r in roster.iterrows():
    z=load_page(r,load_clean=(r.population=='PAIRED_DEVELOPMENT'));obs=z['observed'];c0=z['arms']['C0_INCUMBENT'];ring1=z['ring1'];ring2=z['ring2'];route=z['route']
    if r.population!='PAIRED_DEVELOPMENT':
        # Observable-only page deltas for HITL.
        for arm in ['C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2']:
            changed=z['arms'][arm]!=c0;page_rows.append({'page_key':r.page_key,'page':r.page,'population':r.population,'source_group':r.source_group,'arm':arm,'restored_pixels':int(changed.sum()),'true_print_recovered':np.nan,'annotation_retained':np.nan,'truth_uncertain_retained':np.nan,'false_true_ratio':np.nan,'utility':np.nan,'catastrophic_components':np.nan})
        del z;gc.collect();continue

    clean=z['clean'];clean_print=tolerant_clean_print(clean);observed_dark=obs<DARK_THRESHOLD
    # Truth classes in the selected collar: PRINT if clean-supported; ANNOTATION if dark observed and no clean print; paper/light material separately.
    truth_print=clean_print
    truth_annotation=observed_dark&~clean_print
    truth_uncertain=cv2.dilate(printed_ink(clean).astype(np.uint8),np.ones((3,3),np.uint8))>0
    truth_uncertain=truth_uncertain & ~truth_print
    for arm in ['C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2']:
        selected=(z['arms'][arm]!=c0)
        tp=selected&truth_print;fa=selected&truth_annotation;tu=selected&truth_uncertain&~truth_annotation
        # Component identity follows the selected arm region, not the broad prepared component.
        n,lab,stats,_=cv2.connectedComponentsWithStats(selected.astype(np.uint8),8);cat_count=0
        for cid in range(1,n):
            comp=lab==cid;ctp=int((comp&tp).sum());cfp=int((comp&fa).sum());ctu=int((comp&tu).sum());cat=bool(cfp>=50 or (cfp/max(ctp,1)>1.0 and cfp>=20));cat_count+=int(cat)
            component_rows.append({'page_key':r.page_key,'page':r.page,'source_group':r.source_group,'arm':arm,'component_id':cid,'x':int(stats[cid,cv2.CC_STAT_LEFT]),'y':int(stats[cid,cv2.CC_STAT_TOP]),'w':int(stats[cid,cv2.CC_STAT_WIDTH]),'h':int(stats[cid,cv2.CC_STAT_HEIGHT]),'selected_pixels':int(stats[cid,cv2.CC_STAT_AREA]),'true_print_recovered':ctp,'annotation_retained':cfp,'truth_uncertain_retained':ctu,'false_true_ratio':cfp/max(ctp,1),'utility':ctp-5*cfp-(500 if cat else 0),'catastrophic':cat})
        tpn=int(tp.sum());fan=int(fa.sum());tun=int(tu.sum());utility=tpn-5*fan-500*cat_count
        page_rows.append({'page_key':r.page_key,'page':r.page,'population':r.population,'source_group':r.source_group,'arm':arm,'restored_pixels':int(selected.sum()),'true_print_recovered':tpn,'annotation_retained':fan,'truth_uncertain_retained':tun,'false_true_ratio':fan/max(tpn,1),'false_preservation_rate':fan/max(tpn+fan,1),'utility':utility,'catastrophic_components':cat_count})
        ys,xs=np.where(selected);pixel_rows.append(pd.DataFrame({'page_key':r.page_key,'page':r.page,'source_group':r.source_group,'arm':arm,'y':ys,'x':xs,'ring':np.where(ring1[ys,xs],'RING1','RING2'),'route':route[ys,xs],'truth_class':np.select([truth_print[ys,xs],truth_annotation[ys,xs],tu[ys,xs]],['PRINT','ANNOTATION','UNCERTAIN'],default='OTHER'),'observed':obs[ys,xs],'incumbent':c0[ys,xs],'candidate':z['arms'][arm][ys,xs]}))
    del z;gc.collect()
page_df=pd.DataFrame(page_rows);component_df=pd.DataFrame(component_rows);pixels=pd.concat(pixel_rows,ignore_index=True) if pixel_rows else pd.DataFrame()
page_df.to_csv(TABLES/'results_by_page.csv',index=False);component_df.to_csv(TABLES/'results_by_component.csv',index=False);pixels.to_parquet(TABLES/'paired_selected_pixels.parquet',index=False)
print(page_df.groupby(['population','arm'])[['restored_pixels','true_print_recovered','annotation_retained','utility']].sum(min_count=1))

## 6. Pooled, incremental, source-group and clustered uncertainty summaries

In [ ]:
paired_pages=page_df[page_df.population=='PAIRED_DEVELOPMENT'].copy()
summary=[]
for arm,g in paired_pages.groupby('arm'):
    tp=int(g.true_print_recovered.sum());fp=int(g.annotation_retained.sum());tu=int(g.truth_uncertain_retained.sum());cat=int(g.catastrophic_components.sum())
    rng=np.random.default_rng(SEED);pages=g.page_key.to_numpy();idx=g.set_index('page_key');boot=[]
    for _ in range(BOOTSTRAP_RESAMPLES):
        q=idx.loc[rng.choice(pages,size=len(pages),replace=True)];bt=int(q.true_print_recovered.sum());bf=int(q.annotation_retained.sum());bc=int(q.catastrophic_components.sum());boot.append((bt-5*bf-500*bc,bf/max(bt,1),bf/max(bt+bf,1)))
    b=np.asarray(boot,float);summary.append({'arm':arm,'pages':g.page_key.nunique(),'true_print_recovered':tp,'annotation_retained':fp,'truth_uncertain_retained':tu,'false_true_ratio':fp/max(tp,1),'false_preservation_rate':fp/max(tp+fp,1),'utility':tp-5*fp-500*cat,'catastrophic_components':cat,'benefiting_pages':int((g.true_print_recovered>=10).sum()),'benefiting_source_groups':int(g.loc[g.true_print_recovered>=10,'source_group'].nunique()),'bootstrap_utility_95_low':float(np.quantile(b[:,0],.025)),'bootstrap_utility_95_high':float(np.quantile(b[:,0],.975)),'bootstrap_false_true_95_high':float(np.quantile(b[:,1],.975)),'bootstrap_false_preservation_95_high':float(np.quantile(b[:,2],.975))})
summary_df=pd.DataFrame(summary);summary_df.to_csv(TABLES/'pooled_arm_summary.csv',index=False)
by_group=paired_pages.groupby(['source_group','arm'],as_index=False).agg(pages=('page','nunique'),true_print_recovered=('true_print_recovered','sum'),annotation_retained=('annotation_retained','sum'),truth_uncertain_retained=('truth_uncertain_retained','sum'),utility=('utility','sum'),catastrophic_components=('catastrophic_components','sum'));by_group['false_true_ratio']=by_group.annotation_retained/np.maximum(by_group.true_print_recovered,1);by_group.to_csv(TABLES/'results_by_source_group.csv',index=False)
by_route=pixels.groupby(['arm','ring','route','truth_class'],as_index=False).size().rename(columns={'size':'pixels'}) if len(pixels) else pd.DataFrame();by_route.to_csv(TABLES/'results_by_ring_route_truth.csv',index=False)
# Incremental value of Ring1 = C2 minus C1 at page level.
c1=paired_pages[paired_pages.arm=='C1_PRESERVE_RING2'].set_index('page_key');c2=paired_pages[paired_pages.arm=='C2_PRESERVE_RING1_RING2'].set_index('page_key');inc=(c2[['true_print_recovered','annotation_retained','truth_uncertain_retained','utility']]-c1[['true_print_recovered','annotation_retained','truth_uncertain_retained','utility']]).reset_index();inc.to_csv(TABLES/'incremental_ring1_over_ring2.csv',index=False)
print(summary_df.to_string(index=False))

## 7. Whole-page and focused visual audit outputs

In [ ]:
sns.set_theme(style='whitegrid')
# Pooled trade-off bars.
fig,axes=plt.subplots(1,2,figsize=(14,5))
q=summary_df.set_index('arm').reindex(['C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2']).reset_index();x=np.arange(len(q));axes[0].bar(x-.2,q.true_print_recovered,.4,label='true print recovered');axes[0].bar(x+.2,q.annotation_retained,.4,label='annotation retained');axes[0].set_xticks(x,['C1 Ring2','C2 Ring1+2']);axes[0].set_title('Paired clean-truth benefit and cost');axes[0].legend();axes[1].bar(x,q.utility);axes[1].axhline(0,color='black',linewidth=1);axes[1].set_xticks(x,['C1 Ring2','C2 Ring1+2']);axes[1].set_title('Asymmetric utility: true - 5×false - 500×catastrophe');plt.tight_layout();plt.savefig(VISUAL/'paired_arm_tradeoff.png',dpi=180);plt.close(fig)
# Page scatter.
plt.figure(figsize=(9,7));sns.scatterplot(data=paired_pages,x='true_print_recovered',y='annotation_retained',hue='arm',size='restored_pixels',sizes=(20,260),alpha=.75);mx=max(paired_pages.true_print_recovered.max(),paired_pages.annotation_retained.max(),1);plt.plot([0,mx],[0,mx],'k--');plt.title('Page-level print recovery versus annotation retention');plt.tight_layout();plt.savefig(VISUAL/'paired_page_tradeoff_scatter.png',dpi=180);plt.close()

# Build overlays and output pages.
COL={'RING1':(0,215,255),'RING2':(0,0,255),'CHANGE':(255,0,255),'PRINT':(255,120,0),'ANNOTATION':(0,0,255),'UNCERTAIN':(0,191,255)}
def blend(gray,layers,alpha=.58):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR);out=base.copy()
    for m,c in layers:
        col=np.zeros_like(base);col[:]=c;out[m]=cv2.addWeighted(base,1-alpha,col,alpha,0)[m]
    return out
for _,r in roster.iterrows():
    z=load_page(r,load_clean=(r.population=='PAIRED_DEVELOPMENT'));d=PAGES/r.page_key
    for arm in ['C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2']:
        changed=z['arms'][arm]!=z['arms']['C0_INCUMBENT'];cv2.imwrite(str(d/f'{arm}_authority_overlay.png'),blend(z['observed'],[(z['ring1'],COL['RING1']),(z['ring2'],COL['RING2']),(changed,COL['CHANGE'])]))
        if z['clean'] is not None:
            tp=tolerant_clean_print(z['clean']);ann=(z['observed']<DARK_THRESHOLD)&~tp;cv2.imwrite(str(d/f'{arm}_truth_overlay_posthoc.png'),blend(z['observed'],[(changed&tp,COL['PRINT']),(changed&ann,COL['ANNOTATION'])]))
    if z['clean'] is not None:cv2.imwrite(str(d/'clean.png'),z['clean'])
    del z;gc.collect()

def sheet(rows,name,paired=True):
    rows=rows.head(MAX_CONTACT_PAGES)
    files=[('observed.png','observed'),('C0_INCUMBENT.png','C0'),('C1_PRESERVE_RING2.png','C1 ring2'),('C2_PRESERVE_RING1_RING2.png','C2 ring1+2')]
    if paired:files=[('clean.png','clean truth')]+files
    fig,axes=plt.subplots(len(rows),len(files),figsize=(4.5*len(files),5.3*len(rows)),squeeze=False)
    for i,(_,r) in enumerate(rows.iterrows()):
        for j,(fn,title) in enumerate(files):
            im=cv2.imread(str(PAGES/r.page_key/fn),cv2.IMREAD_UNCHANGED);im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB) if im is not None and im.ndim==3 else im;axes[i,j].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None);axes[i,j].set_title(f'{r.page_key}\n{title}');axes[i,j].axis('off')
    plt.tight_layout();plt.savefig(VISUAL/f'{name}.png',dpi=120,bbox_inches='tight');plt.close(fig)
paired_rank=paired_pages[paired_pages.arm=='C2_PRESERVE_RING1_RING2'].sort_values(['annotation_retained','true_print_recovered'],ascending=False);sheet(paired_rank,'paired_high_annotation_cost_pages',True)
paired_benefit=paired_pages[paired_pages.arm=='C2_PRESERVE_RING1_RING2'].sort_values(['true_print_recovered','utility'],ascending=False);sheet(paired_benefit,'paired_high_print_recovery_pages',True)
hitl_rank=page_df[page_df.population=='HITL_ANNOTATED_ONLY'].sort_values('restored_pixels',ascending=False);sheet(hitl_rank,'hitl_most_changed_pages',False)

## 8. Focused best/worst component crops

In [ ]:
if len(component_df):
    beneficial=component_df.sort_values(['true_print_recovered','utility'],ascending=False).head(MAX_FOCUS_CROPS//2)
    costly=component_df.sort_values(['annotation_retained','selected_pixels'],ascending=False).head(MAX_FOCUS_CROPS//2)
    focus=pd.concat([beneficial.assign(queue='BENEFICIAL'),costly.assign(queue='ANNOTATION_COST')],ignore_index=True).drop_duplicates(['page_key','arm','component_id'])
    focus.to_csv(TABLES/'visual_focus_components.csv',index=False)
    fig,axes=plt.subplots(len(focus),5,figsize=(21,max(6,4.4*len(focus))),squeeze=False)
    for i,(_,r) in enumerate(focus.iterrows()):
        d=PAGES/r.page_key;obs=read_gray(d/'observed.png');clean=read_gray(d/'clean.png');c0=read_gray(d/'C0_INCUMBENT.png');cand=read_gray(d/f'{r.arm}.png');overlay=cv2.imread(str(d/f'{r.arm}_truth_overlay_posthoc.png'));overlay=cv2.cvtColor(overlay,cv2.COLOR_BGR2RGB)
        x0=max(0,int(r.x)-CROP_MARGIN);y0=max(0,int(r.y)-CROP_MARGIN);x1=min(obs.shape[1],int(r.x+r.w)+CROP_MARGIN);y1=min(obs.shape[0],int(r.y+r.h)+CROP_MARGIN)
        ims=[obs[y0:y1,x0:x1],clean[y0:y1,x0:x1],c0[y0:y1,x0:x1],cand[y0:y1,x0:x1],overlay[y0:y1,x0:x1]]
        for j,(im,title) in enumerate(zip(ims,['observed','clean','C0','candidate','truth overlay'])):
            axes[i,j].imshow(im,cmap='gray' if im.ndim==2 else None);axes[i,j].set_title(f'{r.queue} {r.page_key}\n{r.arm} {title}\nP={r.true_print_recovered} A={r.annotation_retained} U={r.utility}');axes[i,j].axis('off')
    plt.tight_layout();plt.savefig(VISUAL/'focused_best_and_worst_components.png',dpi=145,bbox_inches='tight');plt.close(fig)

## 9. Verdict record and final upload package

In [ ]:
# Descriptive verdict only: no post-hoc production gate is invented here.
arm_records=summary_df.set_index('arm').to_dict(orient='index') if len(summary_df) else {}
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook75-v88-collar-preservation-counterfactual','source_notebook74_run':str(SOURCE),'prediction_lock_id':lock['lock_id'],'development_only':True,'promotion_authority':False,'arms':arm_records,'interpretation':{'C1':'marginal effect of restoring Ring 2','C2':'total effect of restoring Rings 1 and 2','C2_minus_C1':'marginal effect of restoring Ring 1'},'decision_rule':'Evidence only. Choose no winner automatically. Review pooled utility, annotation cost, catastrophes, source stability and visual queues.'}
atomic_json(OUT/'notebook75_verdict.json',verdict)
lines=['NOTEBOOK 75 — FROZEN V8.8 COLLAR-PRESERVATION COUNTERFACTUAL','',f'Source Notebook 74: {SOURCE}',f'Prediction lock: {lock["lock_id"]}',f'Pages: {len(roster)}',f'Paired pages: {(roster.population=="PAIRED_DEVELOPMENT").sum()}',f'HITL pages: {(roster.population=="HITL_ANNOTATED_ONLY").sum()}','','PAIRED CLEAN-TRUTH ARM SUMMARY',summary_df.to_string(index=False),'','Interpretation: C1 restores Ring 2 only. C2 restores Ring 1 and Ring 2. C2-C1 is the marginal Ring 1 effect.','No arm is promoted automatically. The incumbent remains unchanged.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))
required=[OUT/'executive_summary.txt',OUT/'notebook75_verdict.json',PRED/'prediction_lock.json',PRED/'page_output_hashes.csv',TABLES/'pooled_arm_summary.csv',TABLES/'results_by_page.csv',TABLES/'results_by_source_group.csv',TABLES/'incremental_ring1_over_ring2.csv',TABLES/'results_by_component.csv',VISUAL/'paired_arm_tradeoff.png',VISUAL/'paired_page_tradeoff_scatter.png',VISUAL/'paired_high_annotation_cost_pages.png',VISUAL/'paired_high_print_recovery_pages.png',VISUAL/'hitl_most_changed_pages.png']
if (VISUAL/'focused_best_and_worst_components.png').exists():required.append(VISUAL/'focused_best_and_worst_components.png')
for p in required:require(p.exists(),f'Missing upload artefact: {p}')
for p in required:
    target=UPLOAD/p.name;shutil.copy2(p,target)
manifest=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file():manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 75 collar-preservation counterfactual. Start with executive_summary.txt, pooled_arm_summary.csv, the paired trade-off plots and the best/worst contact sheets. C0/C1/C2 page images remain in the run pages directory.\n')
with zipfile.ZipFile(UPLOAD/'notebook75_collar_counterfactual_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip':z.write(p,p.name)
print('Bundle:',UPLOAD/'notebook75_collar_counterfactual_bundle.zip')

## Reading the result

1. **C1 versus C0** measures the value and cost of preserving Ring 2.
2. **C2 versus C1** measures the incremental value and cost of also preserving Ring 1.
3. Do not compare HITL observable changes with paired clean-truth error rates as if they were the same endpoint.
4. A favourable pooled result is not enough if annotation retention or catastrophic concentrations are unstable by source group or page.
5. `v8_8_operator.py` remains untouched. Any eventual operator change requires a separate documented decision after this counterfactual is reviewed.